# Eye tracking · 3.1 · Hurst dataframe



In [2]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))


In [3]:
import pandas as pd
import scipy.io as sio
import numpy as np
import os

In [7]:
# parameters

data_type = "pupil_size"

data_path = f"{base_path}/RESULTS/pupillometry/{data_type}/hurst/raw/decimated_2s/FIN_hurst_exponents_D4_modwt_1_5.mat"
output_path = f"{base_path}/RESULTS/pupillometry/{data_type}/hurst/dataframes/"
os.makedirs(output_path, exist_ok=True)
output_name = f"df_hurst_{data_type}_FIN_D4_modwt_1_5_decimated_2s.csv"

task_names = ["0Back", "1Back", "2Back"]

In [8]:
# functions

def find_modality(order_idx):
    if order_idx in [1, 3, 5]:
        return 'fMRI'
    elif order_idx in [2, 4, 6]:
        return 'fMRS'

In [9]:
# load data
data = sio.loadmat(data_path)
hurst_data = data['hursts']

# reshape data to long format
df_long = []
for sub_idx in range(hurst_data.shape[0]):
    for order_idx in range(hurst_data.shape[1]):
        if hurst_data[sub_idx, order_idx] > 0:
            df_long.append({
                'Hurst': hurst_data[sub_idx, order_idx],
                'Task': task_names[order_idx // 2],
                'Modality': find_modality(order_idx + 1),
                'Subject': sub_idx + 1,
                'order': order_idx + 1
            })

df_long = pd.DataFrame(df_long)

# Remove outliers using MAD method for each combination of Modality and task
df_long_clean = df_long.copy()

rows_to_keep_fin = []
for Modality in df_long_clean['Modality'].unique():
    for task in df_long_clean['Task'].unique():
        mask = (df_long_clean['Modality'] == Modality) & (df_long_clean['Task'] == task)
        subset = df_long_clean.loc[mask, 'Hurst'].values
        
        if len(subset) > 0:
            median_val = np.median(subset)
            mad_val = np.median(np.abs(subset - median_val))
            lower = median_val - 2 * mad_val
            upper = median_val + 2 * mad_val
            
            outlier_mask = (subset >= lower) & (subset <= upper)
            rows_to_keep_fin.append(df_long_clean[mask].iloc[outlier_mask].index)

df_long_clean = df_long_clean.loc[np.concatenate(rows_to_keep_fin)]
df_long_clean = df_long_clean.reset_index(drop=True)

print(f"{len(df_long)} -> {len(df_long_clean)} rows ({len(df_long) - len(df_long_clean)} removed)")

# save cleaned dataframe
os.makedirs(output_path, exist_ok=True)
df_long_clean.to_csv(f"{output_path}/{output_name}", index=False)

198 -> 157 rows (41 removed)
